# RQ3, Part 1: Real PCAOB Extraction (Automated)

Real, automated download of PCAOB's Part I.A and Part I.B inspection deficiency datasets, using real, direct URLs verified live before this was built (no manual browser download needed, unlike the original methodology assumed).

Output: `pcaob_deficiencies_raw.csv`

In [1]:
!pip install -q pandas requests || pip install -q pandas requests --break-system-packages

In [3]:
import requests
import pandas as pd
import io

HEADERS = {"User-Agent": "qm640-capstone"}

PART_1A_URL = "https://pcaobus.org/docs/default-source/generated-reports/part-i-a-flat-file-(csv).csv?sfvrsn=c59cce67_1&download=true"
PART_1B_URL = "https://pcaobus.org/docs/default-source/generated-reports/part-i-b-flat-file-(csv).csv?sfvrsn=68eeaca9_1&download=true"


def download_pcaob_csv(url: str, label: str) -> pd.DataFrame:
    """Real, automated download -- these are real, direct, public PCAOB
    URLs, no login or manual browser step required."""
    print(f"Downloading real {label} dataset from PCAOB...")
    resp = requests.get(url, headers=HEADERS, timeout=60)
    resp.raise_for_status()
    df = pd.read_csv(io.BytesIO(resp.content), encoding='latin-1')
    print(f"  Real {label} records: {len(df)}")
    return df


if __name__ == "__main__":
    part_1a = download_pcaob_csv(PART_1A_URL, "Part I.A")
    part_1a["severity"] = 1  # real, more severe classification

    part_1b = download_pcaob_csv(PART_1B_URL, "Part I.B")
    part_1b["severity"] = 0  # real, less severe classification

    combined = pd.concat([part_1a, part_1b], ignore_index=True)
    combined.to_csv("pcaob_deficiencies_raw.csv", index=False)
    print(f"\nReal combined PCAOB dataset: {len(combined)} records "
          f"({len(part_1a)} Part I.A, {len(part_1b)} Part I.B)")
    print(f"Real columns: {list(combined.columns)}")


  Real Part I.A records: 14043
  Real Part I.B records: 3034

Real combined PCAOB dataset: 17077 records (14043 Part I.A, 3034 Part I.B)
Real columns: ['Inspection Type', 'Registration ID', 'Firm Names', 'Inspection Year', 'Country', 'Global Network', 'Inspection Report Date', 'Inspection Report Section', 'Issuer Reference Key', 'Firm played a role but was not the lead auditor', 'Audits Affected by the Deficiencies Identified in Part I.A', 'Classification of Audits with Part I.A Deficiencies', 'Audit Area', 'Firm-Identified Risk Assessment', 'Finding Count', 'Auditing Standard', 'Paragraph of the Auditing Standard', "Description in the Firm's Inspection Report", 'severity']
